# Ensemble of Specialized Mixture of Experts (MoE) for NLI
This notebook implements a state-of-the-art Natural Language Inference pipeline combining:
1. **T5 Data Augmentation**: Generating synthetic hypotheses to improve robustness.
2. **POS-Specialized MoE**: A custom architecture with experts for Semantics, Entities, Actions, and Logic.
3. **Model Ensembling**: Averaging predictions from **DeBERTa-v3** and **ModernBERT** backbones.

In [1]:
!pip install datasets transformers torch spacy pandas numpy scikit-learn
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 28.2 MB/s eta 0:00:00m eta 0:00:010:01:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [2]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import spacy
from datasets import Dataset
from transformers import (
    AutoTokenizer, 
    AutoModel, 
    T5Tokenizer, 
    T5ForConditionalGeneration, 
    TrainingArguments, 
    Trainer
)
from sklearn.metrics import f1_score, accuracy_score

nlp = spacy.load("en_core_web_sm", disable=["ner"])
device = "cuda" if torch.cuda.is_available() else "cpu"

## 1. Data Augmentation (T5)
Use a generative transformer model to expand the available dataset by paraphrasing the existing examples.

In [4]:
def augment_dataset(df):
    generativeModel = "google/flan-t5-base"
    t5Tokenizer = T5Tokenizer.from_pretrained(generativeModel)
    generator = T5ForConditionalGeneration.from_pretrained(generativeModel).to(device)

    sampleSize = 100
    generator.eval()
    generatedExamples = []
    subset = df.sample(n=sampleSize)

    for _, ex in subset.iterrows():
        prompt = f"make a sentence that means this: {ex['hypothesis']}"
        inputs = t5Tokenizer(prompt, return_tensors="pt").to(device)
        outputs = generator.generate(**inputs, max_length=64)
        gen_hyp = t5Tokenizer.decode(outputs[0], skip_special_tokens=True)

        promptP = f"make a sentence that means this: {ex['premise']}"
        inputsP = t5Tokenizer(promptP, return_tensors="pt").to(device)
        outputsP = generator.generate(**inputsP, max_length=64)
        gen_prem = t5Tokenizer.decode(outputsP[0], skip_special_tokens=True)

        generatedExamples.append({"premise": gen_prem, "hypothesis": gen_hyp, "label": ex["label"]})

    return pd.concat([df, pd.DataFrame(generatedExamples)]).reset_index(drop=True)

trainingDataframe = pd.read_csv("training_data/NLI/trainSmall.csv")
devDataframe = pd.read_csv("training_data/NLI/devSmall.csv")
trainingDataframeWithGenerated = augment_dataset(trainingDataframe)


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


## 2. Multi-Backbone MoE Architecture
Each model in the ensemble utilizes a Mixture of Experts layer. This layer routes features to specialized heads based on POS-filtered text (Nouns for Entities, Verbs for Actions) and manual Logic features.

In [5]:
class POSSpecializedMoE(nn.Module):
    def __init__(self, model_name, num_labels=2):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name).float() 
        hidden_size = self.encoder.config.hidden_size
        
        num_logic_features = 4
        self.semantic_expert = nn.Linear(hidden_size, hidden_size)
        self.entity_expert = nn.Linear(hidden_size, hidden_size)
        self.action_expert = nn.Linear(hidden_size, hidden_size)
        self.logic_expert = nn.Linear(num_logic_features, hidden_size)
        
        self.logic_norm = nn.LayerNorm(num_logic_features)
        self.gating = nn.Sequential(
            nn.Linear(hidden_size + num_logic_features, 128),
            nn.ReLU(),
            nn.Linear(128, 4),
            nn.Softmax(dim=-1)
        )
        
        self.classifier = nn.Linear(hidden_size, num_labels)
        self.loss_fn = nn.CrossEntropyLoss()

    def forward(self, input_ids, attention_mask, entity_ids=None, action_ids=None, logic_features=None, labels=None, **kwargs):
            self.encoder.float() 
            
            outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
            cls_output = outputs.last_hidden_state[:, 0, :].float() 

            if logic_features is None:
                logic_features = torch.zeros((cls_output.size(0), 4), device=cls_output.device)
            
            logic_features = logic_features.float()
            
            norm_logic = self.logic_norm(logic_features)

            e_sem = torch.tanh(self.semantic_expert(cls_output))
            e_ent = torch.tanh(self.entity_expert(cls_output))
            e_act = torch.tanh(self.action_expert(cls_output))
            e_log = torch.tanh(self.logic_expert(norm_logic))

            gate_input = torch.cat([cls_output, norm_logic], dim=-1)
            gate_weights = self.gating(gate_input) 

            experts = torch.stack([e_sem, e_ent, e_act, e_log], dim=1)
            moe_output = torch.bmm(gate_weights.unsqueeze(1), experts).squeeze(1)

            logits = self.classifier(moe_output)

            loss = None
            if labels is not None:
                loss = self.loss_fn(logits, labels)

            return {"loss": loss, "logits": logits} if loss is not None else {"logits": logits}

## 3. Preprocessing and Feature Extraction
We extract linguistic features using Spacy to feed the MoE gating and specialized experts.

In [6]:
# Extract words based on a specific POS
def get_pos_filtered_text(text, pos_tags):
    # Generate POS tags
    doc = nlp(str(text))
    # Only select those which have been specified
    tokens = [token.text for token in doc if token.pos_ in pos_tags]
    return " ".join(tokens) if tokens else "none"

def extract_logic_features(premise, hypothesis):
    # Produce POS tags for both the premise and hypothesis
    p_doc, h_doc = nlp(str(premise)), nlp(str(hypothesis))
    # Count how many negations we see between the 2 sentences - a mismatch indicates contradiction
    neg_p = sum(1 for t in p_doc if t.dep_ == "neg")
    neg_h = sum(1 for t in h_doc if t.dep_ == "neg")
    # Removes punctuation and stopwords
    p_set = {t.lemma_.lower() for t in p_doc if not t.is_stop and not t.is_punct}
    h_set = {t.lemma_.lower() for t in h_doc if not t.is_stop and not t.is_punct}
    # Determines the overlap between sentences using Jaccard Similarity
    overlap = len(p_set & h_set) / len(p_set | h_set) if (p_set | h_set) else 0.0
    return [float(neg_p), float(neg_h), float(abs(neg_p - neg_h)), float(overlap)]

# Factory to handle HuggingFace map function
def make_preprocess_fn(tokenizer):
    def preprocess(example):
        main_enc = tokenizer(example["premise"], example["hypothesis"], truncation=True, padding="max_length", max_length=128)
        
        # Extract the relevant sentences for each expert
        p_nouns = get_pos_filtered_text(example["premise"], ["NOUN", "PROPN"])
        h_nouns = get_pos_filtered_text(example["hypothesis"], ["NOUN", "PROPN"])
        p_verbs = get_pos_filtered_text(example["premise"], ["VERB"])
        h_verbs = get_pos_filtered_text(example["hypothesis"], ["VERB"])
        
        # Encoders for the 2 experts
        ent_enc = tokenizer(p_nouns, h_nouns, truncation=True, padding="max_length", max_length=128)
        act_enc = tokenizer(p_verbs, h_verbs, truncation=True, padding="max_length", max_length=128)
        
        # Returns the sentences which each expert relies on
        return {
            "input_ids": main_enc["input_ids"],
            "attention_mask": main_enc["attention_mask"],
            "entity_ids": ent_enc["input_ids"],
            "action_ids": act_enc["input_ids"],
            "logic_features": extract_logic_features(example["premise"], example["hypothesis"]),
            "label": int(example["label"])
        }
    return preprocess

## 4. Training and Ensembling
We train two separate MoE models (DeBERTa and ModernBERT) and then ensemble their outputs by averaging the logits.

In [9]:
from transformers import EarlyStoppingCallback, DefaultDataCollator

# Dictionary mapping backbone to its specific optimal search results
optimal_configs = {
    "deberta": {
        "lr": 3e-5, 
        "batch_size": 16, 
        "epochs": 12 # Based on your previous run's peak
    },
    "modernbert": {
        "lr": 3e-5, 
        "batch_size": 8, 
        "epochs": 8 # Based on your previous run's peak
    }
}

backbones = {
    "deberta": "microsoft/deberta-v3-small",
    "modernbert": "answerdotai/ModernBERT-base"
}

all_logits = {}

for name, path in backbones.items():
    print(f"\n--- Training Optimized MoE: {name} ---")
    
    # Setup for specific backbone
    config = optimal_configs[name]
    tokenizer = AutoTokenizer.from_pretrained(path)
    model = POSSpecializedMoE(path).to(device)
    
    prep_fn = make_preprocess_fn(tokenizer)
    train_ds = Dataset.from_pandas(trainingDataframeWithGenerated).map(prep_fn)
    dev_ds = Dataset.from_pandas(devDataframe).map(prep_fn)
    
    args = TrainingArguments(
        output_dir=f"moe_{name}_optimized",
        learning_rate=config["lr"],
        per_device_train_batch_size=config["batch_size"],
        num_train_epochs=config["epochs"],
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True, # Rolls back to highest accuracy checkpoint
        metric_for_best_model="accuracy",
        fp16=False,
        bf16=False,
        report_to="none"
    )

    trainer = Trainer(
        model=model, 
        args=args, 
        train_dataset=train_ds, 
        eval_dataset=dev_ds,
        data_collator=DefaultDataCollator(),
        # Stops if no improvement for 3 evaluations to save time
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
        compute_metrics=lambda p: {"accuracy": accuracy_score(p.label_ids, np.argmax(p.predictions, axis=1))}
    )
    
    trainer.train()
    
    # Store the best predictions for the ensemble
    preds = trainer.predict(dev_ds)
    all_logits[name] = preds.predictions

# Ensemble Logic: Average Logits
final_logits = (all_logits["deberta"] + all_logits["modernbert"]) / 2
final_preds = np.argmax(final_logits, axis=1)

print("\n--- Final Optimized Ensemble Metrics ---")
print(f"Accuracy: {accuracy_score(devDataframe['label'], final_preds):.4f}")
print(f"Macro F1: {f1_score(devDataframe['label'], final_preds, average='macro'):.4f}")


--- Training Optimized MoE: deberta ---


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.classifier.bias        | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Map:   0%|          | 0/1099 [00:00<?, ? examples/s]

Map:   0%|          | 0/999 [00:00<?, ? examples/s]

...

ModernBertModel LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     |  | 
------------------+------------+--+-
head.dense.weight | UNEXPECTED |  | 
decoder.bias      | UNEXPECTED |  | 
head.norm.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Map:   0%|          | 0/1099 [00:00<?, ? examples/s]

Map:   0%|          | 0/999 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.673396,0.575576
2,No log,0.673050,0.573574
3,No log,0.681862,0.572573
4,0.660832,0.677458,0.573574



--- Final Optimized Ensemble Metrics ---
Accuracy: 0.7107
Macro F1: 0.7092


In [ ]:
# import itertools
# from transformers import DefaultDataCollator

# # 1. Define the search space
# param_grid = {
#     "learning_rate": [1e-5, 3e-5],
#     "per_device_train_batch_size": [8, 16],
#     "num_train_epochs": [2, 3]
# }

# # Generate all combinations of hyperparameters
# keys, values = zip(*param_grid.items())
# combinations = [dict(zip(keys, v)) for v in itertools.product(*values)]

# results_file = "hyperparameter_results.txt"

# with open(results_file, "w") as f:
#     f.write("Backbone, LR, BatchSize, Epochs, Accuracy, F1\n")

# print(f"Starting search over {len(combinations)} combinations per backbone...")

# for name, path in backbones.items():
#     tokenizer = AutoTokenizer.from_pretrained(path)
#     prep_fn = make_preprocess_fn(tokenizer)
    
#     # Pre-process datasets once per backbone to save time
#     train_ds = Dataset.from_pandas(trainingDataframeWithGenerated).map(prep_fn)
#     dev_ds = Dataset.from_pandas(devDataframe).map(prep_fn)
    
#     for params in combinations:
#         print(f"\nTesting {name} with: {params}")
        
#         # Re-initialize model for each run to reset weights
#         model = POSSpecializedMoE(path).to(device)
        
#         args = TrainingArguments(
#             output_dir=f"search_{name}",
#             learning_rate=params["learning_rate"],
#             per_device_train_batch_size=params["per_device_train_batch_size"],
#             num_train_epochs=params["num_train_epochs"],
#             eval_strategy="no", # Speed up search by only evaluating at the end
#             save_strategy="no",
#             fp16=False,
#             bf16=False,
#             report_to="none"
#         )
        
#         trainer = Trainer(
#             model=model,
#             args=args,
#             train_dataset=train_ds,
#             data_collator=DefaultDataCollator()
#         )
        
#         trainer.train()
        
#         # Evaluate
#         preds_output = trainer.predict(dev_ds)
#         preds = np.argmax(preds_output.predictions, axis=1)
        
#         acc = accuracy_score(devDataframe['label'], preds)
#         f1 = f1_score(devDataframe['label'], preds, average='macro')
        
#         # Store results
#         result_line = f"{name}, {params['learning_rate']}, {params['per_device_train_batch_size']}, {params['num_train_epochs']}, {acc:.4f}, {f1:.4f}"
#         print(f"Result: {result_line}")
        
#         with open(results_file, "a") as f:
#             f.write(result_line + "\n")

# print(f"\nSearch complete. Results saved to {results_file}")

In [ ]:
# from transformers import EarlyStoppingCallback, DefaultDataCollator

# # Define the optimal parameters found from search
# optimal_configs = {
#     "deberta": {"lr": 3e-5, "batch_size": 16},
#     "modernbert": {"lr": 3e-5, "batch_size": 8}
# }

# final_results_file = "final_training_results.txt"

# # Prepare the log file
# with open(final_results_file, "w") as f:
#     f.write("Backbone, Epochs_Completed, Final_Accuracy, Final_F1\n")

# all_final_logits = {}

# for name, path in backbones.items():
#     print(f"\n--- Final Long-Run Training: {name} ---")
    
#     config = optimal_configs[name]
#     tokenizer = AutoTokenizer.from_pretrained(path)
#     prep_fn = make_preprocess_fn(tokenizer)
    
#     train_ds = Dataset.from_pandas(trainingDataframeWithGenerated).map(prep_fn)
#     dev_ds = Dataset.from_pandas(devDataframe).map(prep_fn)
    
#     model = POSSpecializedMoE(path).to(device)
    
#     training_args = TrainingArguments(
#         output_dir=f"final_model_{name}",
#         learning_rate=config["lr"],
#         per_device_train_batch_size=config["batch_size"],
#         num_train_epochs=50,
#         eval_strategy="epoch",      # Required for EarlyStopping
#         save_strategy="epoch",      # Required for EarlyStopping
#         load_best_model_at_end=True, # Ensure we keep the best version
#         metric_for_best_model="accuracy",
#         fp16=False,
#         bf16=False,
#         report_to="none",
#         logging_steps=10
#     )
    
#     trainer = Trainer(
#         model=model,
#         args=training_args,
#         train_dataset=train_ds,
#         eval_dataset=dev_ds,
#         data_collator=DefaultDataCollator(),
#         # Stopping if accuracy doesn't improve for 5 epochs
#         callbacks=[EarlyStoppingCallback(early_stopping_patience=5)],
#         compute_metrics=lambda p: {
#             "accuracy": accuracy_score(p.label_ids, np.argmax(p.predictions, axis=1))
#         }
#     )
    
#     # Train and save results
#     train_result = trainer.train()
    
#     # Final Predict
#     preds_output = trainer.predict(dev_ds)
#     all_final_logits[name] = preds_output.predictions
    
#     final_acc = accuracy_score(devDataframe['label'], np.argmax(all_final_logits[name], axis=1))
#     final_f1 = f1_score(devDataframe['label'], np.argmax(all_final_logits[name], axis=1), average='macro')
    
#     # Record to file
#     with open(final_results_file, "a") as f:
#         f.write(f"{name}, {train_result.global_step}, {final_acc:.4f}, {final_f1:.4f}\n")
    
#     print(f"Finished {name}. Best Accuracy: {final_acc:.4f}")

# # --- Final Ensemble Step ---
# ensemble_logits = (all_final_logits["deberta"] + all_final_logits["modernbert"]) / 2
# ensemble_preds = np.argmax(ensemble_logits, axis=1)

# print("\n--- Final Optimized Ensemble Metrics ---")
# print(f"Accuracy: {accuracy_score(devDataframe['label'], ensemble_preds):.4f}")
# print(f"Macro F1: {f1_score(devDataframe['label'], ensemble_preds, average='macro'):.4f}")

In [8]:
import os
import torch
import numpy as np
from transformers import Trainer, TrainingArguments, EarlyStoppingCallback, DefaultDataCollator, AutoTokenizer
from sklearn.metrics import accuracy_score, f1_score
from datasets import Dataset

# 1. Define the base directory for your ensemble
ensemble_save_path = "nli_ensemble_model2"
os.makedirs(ensemble_save_path, exist_ok=True)

all_logits = {}

for name, path in backbones.items():
    print(f"\n" + "="*50)
    print(f"--- Training Optimized MoE: {name} ---")
    print(f"="*50)
    
    # Setup for specific backbone
    config = optimal_configs[name]
    tokenizer = AutoTokenizer.from_pretrained(path)
    model = POSSpecializedMoE(path).to(device)
    
    prep_fn = make_preprocess_fn(tokenizer)
    train_ds = Dataset.from_pandas(trainingDataframeWithGenerated).map(prep_fn)
    dev_ds = Dataset.from_pandas(devDataframe).map(prep_fn)
    
    args = TrainingArguments(
        output_dir=f"moe_{name}_optimized",
        learning_rate=config["lr"],
        per_device_train_batch_size=config["batch_size"],
        num_train_epochs=config["epochs"],
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,  # This ensures 'model' is the best version after training
        metric_for_best_model="accuracy",
        fp16=False,
        bf16=False,
        report_to="none"
    )

    trainer = Trainer(
        model=model, 
        args=args, 
        train_dataset=train_ds, 
        eval_dataset=dev_ds,
        data_collator=DefaultDataCollator(),
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
        compute_metrics=lambda p: {"accuracy": accuracy_score(p.label_ids, np.argmax(p.predictions, axis=1))}
    )
    
    # Run training
    trainer.train()
    
    # --- SAVING LOGIC ---
    # Create specific subfolder: nli_ensemble_model/deberta/ or nli_ensemble_model/modernbert/
    specific_save_dir = os.path.join(ensemble_save_path, name)
    os.makedirs(specific_save_dir, exist_ok=True)
    
    print(f"--> Saving {name} to {specific_save_dir}...")
    
    # Save the model weights (state_dict)
    torch.save(model.state_dict(), os.path.join(specific_save_dir, "moe_weights.pt"))
    
    # Save the tokenizer (so the inference file can load it via .from_pretrained)
    tokenizer.save_pretrained(specific_save_dir)
    # --------------------
    
    # Store the best predictions for the ensemble
    preds = trainer.predict(dev_ds)
    all_logits[name] = preds.predictions

# Ensemble Logic: Average Logits
final_logits = (all_logits["deberta"] + all_logits["modernbert"]) / 2
final_preds = np.argmax(final_logits, axis=1)

print("\n" + "="*50)
print("--- Final Optimized Ensemble Metrics ---")
print(f"Accuracy: {accuracy_score(devDataframe['label'], final_preds):.4f}")
print(f"Macro F1: {f1_score(devDataframe['label'], final_preds, average='macro'):.4f}")
print("="*50)


--- Training Optimized MoE: deberta ---


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.classifier.bias        | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Map:   0%|          | 0/1099 [00:00<?, ? examples/s]

Map:   0%|          | 0/999 [00:00<?, ? examples/s]

...

Map:   0%|          | 0/1099 [00:00<?, ? examples/s]

Map:   0%|          | 0/999 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.678528,0.572573
2,No log,0.680404,0.573574
3,No log,0.684423,0.569570
4,0.674409,0.680171,0.566567
5,0.674409,0.673181,0.566567


--> Saving modernbert to nli_ensemble_model2/modernbert...



--- Final Optimized Ensemble Metrics ---
Accuracy: 0.6767
Macro F1: 0.6708
